<a href="https://colab.research.google.com/github/srinivasulu-2026/my-first-repo/blob/main/Surge_Pricing_workshop_30_Aug_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
rng = np.random.default_rng(23)

N = 4000

zones = ["Airport", "Tech Park", "Railway Station", "Mall District", "Old City", "Residential"]
zone_w = [0.12, 0.22, 0.10, 0.18, 0.16, 0.22]
zone_effect = {"Airport": 0.15, "Tech Park": 0.10, "Railway Station": 0.05,
               "Mall District": 0.0, "Old City": -0.05, "Residential": -0.10}

rows = []
base_date = datetime(2024, 6, 1)

for i in range(N):
    ride_id = f"RIDE{20000+i}"
    day_offset = int(rng.integers(0, 60))
    hour = int(rng.integers(0, 24))
    ts = base_date + timedelta(days=day_offset, hours=hour, minutes=int(rng.integers(0, 60)))
    is_weekend = int(ts.weekday() >= 5)
    is_night = int(hour >= 21 or hour < 6)

    zone = rng.choice(zones, p=zone_w)

    distance_km = float(np.clip(rng.gamma(3, 2.2), 0.5, 30))
    duration_min = float(np.clip(distance_km * rng.normal(3.2, 0.5) + rng.normal(0, 3), 3, 90))
    temperature_c = float(np.clip(rng.normal(27, 4), 14, 40))

    rain_base_prob = 0.22 + (0.08 if is_night else 0)
    is_raining = int(rng.random() < rain_base_prob)
    rainfall_mm = float(np.clip(rng.gamma(2, 6), 0, 45)) if is_raining else 0.0

    active_drivers_nearby = int(np.clip(rng.poisson(22), 3, 60))
    ride_requests_nearby = int(np.clip(rng.poisson(18 + (6 if is_night else 0) + (5 if is_raining else 0)), 2, 70))

    surge = 1.0
    surge += 0.30 * is_raining
    surge += 0.008 * rainfall_mm
    surge += 0.22 * is_night
    surge += 0.25 * (is_raining * is_night)
    surge += 0.10 * is_weekend
    surge += 0.020 * ride_requests_nearby
    surge -= 0.018 * active_drivers_nearby
    surge += zone_effect[zone]
    surge += rng.normal(0, 0.15)
    surge = round(float(np.clip(surge, 1.0, 4.0)), 2)

    base_fare = 60 + 12 * distance_km + 2 * duration_min
    fare_amount = max(round(base_fare * surge + rng.normal(0, 8), 2), 40)

    rows.append({
        "ride_id": ride_id, "request_timestamp": ts.strftime("%Y-%m-%d %H:%M"),
        "hour": hour, "is_night": is_night, "is_weekend": is_weekend, "zone": zone,
        "distance_km": round(distance_km, 2), "duration_min": round(duration_min, 1),
        "is_raining": is_raining, "rainfall_mm": round(rainfall_mm, 1),
        "temperature_c": round(temperature_c, 1),
        "active_drivers_nearby": active_drivers_nearby,
        "ride_requests_nearby": ride_requests_nearby,
        "fare_amount": fare_amount, "surge_multiplier": surge,
    })

df = pd.DataFrame(rows)
df.to_csv("ride_surge.csv", index=False)
print(f"✅ ride_surge.csv — {len(df):,} rows, {df.shape[1]} columns")


✅ ride_surge.csv — 4,000 rows, 15 columns
